In [54]:
import math
import torch
import random
from value import Value
from graph import draw_dot

In [13]:
x1 = torch.Tensor([2.0]).double()
x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()
x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()
w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()
w2.requires_grad = True
b = torch.Tensor([6.8813725323498345]).double()
b.requires_grad = True
n = x1 * w1 + x2 * w2 + b
o = torch.tanh(n)

In [14]:
print(o.data.item())
o.backward()

0.7071062135674336


In [15]:
print("x1", x1.grad.item())
print("x2", x2.grad.item())
print("w1", w1.grad.item())
print("w2", w2.grad.item())

x1 -1.5000024082029806
x2 0.5000008027343269
w1 1.0000016054686538
w2 0.0


## Neural net (Multilayer perceptron)


In [41]:
# nin - number of weights, nout - how many neuron

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    # forward pass
    def __call__(self, x):
        # sig(w * x) + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    # forward pass
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
class MLP: # multi layer perceptron
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    # forward pass
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [70]:
# x = [2.0, 3.0]
# n = Layer(2, 3) # Neuron(2) three times
# n(x)

x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
"""
input: 3 features
layer1: 4 neurons
layer2: 4 neurons
layer3: 1 neuron
"""

n(x)

Value(data=0.7882158390879597)

## Small Dataset


In [69]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]
ys = [1.0, -1.0, -1.0, 1.0]  # desired  targets
ypred = [n(x) for x in xs]
ypred

[Value(data=-0.36311382571795753),
 Value(data=-0.6064086346659443),
 Value(data=-0.4132685304856745),
 Value(data=-0.4530904754155657)]

In [73]:
loss = [(yout - ygt)**2 for ygt, yout in zip(ys, ypred)]
loss

[Value(data=1.8580793018634465),
 Value(data=0.1549141628655261),
 Value(data=0.3442538173184399),
 Value(data=2.1114719297434346)]